In [1]:
import subprocess, os

wheel_dir = "/kaggle/input/datasets/arun612/sacrebleu-wheel"
wheels = sorted([
    os.path.join(wheel_dir, f)
    for f in os.listdir(wheel_dir)
    if f.endswith(".whl")
])
for w in wheels:
    print(f"Installing {os.path.basename(w)}...")
    subprocess.run(["pip", "install", "-q", w, "--no-deps"], check=True)
print("Done.")

Installing colorama-0.4.6-py2.py3-none-any.whl...
Installing lxml-6.0.2-cp312-cp312-manylinux_2_26_x86_64.manylinux_2_28_x86_64.whl...
Installing portalocker-3.2.0-py3-none-any.whl...
Installing regex-2026.2.28-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl...
Installing sacrebleu-2.6.0-py3-none-any.whl...
Installing tabulate-0.10.0-py3-none-any.whl...
Done.


In [2]:
# ===========================================================================
# CELL 1 — Imports
# ===========================================================================

import os, gc, re, json, math, random, logging, warnings
from pathlib import Path
from dataclasses import dataclass, field
from contextlib import nullcontext
from typing import List, Tuple

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import sacrebleu

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")

def _cuda_bf16_supported():
    if not torch.cuda.is_available(): return False
    try: return bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
    except: return False

def _bf16_ctx(device, enabled):
    if enabled and device.type == "cuda" and _cuda_bf16_supported():
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"GPU Mem  : {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")
    print(f"BF16     : {_cuda_bf16_supported()}")

PyTorch  : 2.9.0+cu126
CUDA     : True
GPU      : Tesla T4
GPU Mem  : 15.64 GB
BF16     : True


In [3]:
# ===========================================================================
# CELL 2 — Configuration
# ===========================================================================

@dataclass
class Config:
    output_dir: str = "/kaggle/working/"

    # ── Model paths — auto-detected below ────────────────────────────────
    model_a_path: str = "/kaggle/input/datasets/assiaben/final-byt5/byt5-akkadian-optimized-34x"
    model_b_path: str = "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1"
    # Third model from LB35.9 notebook (vitorhugobarbedo)
    model_c_path: str = "/kaggle/input/datasets/vitorhugobarbedo/model-final-2026"

    max_input_length : int   = 512
    max_new_tokens   : int   = 384
    batch_size       : int   = 2
    num_workers      : int   = 2
    num_buckets      : int   = 6

    # beam search
    num_beam_cands   : int   = 4
    num_beams        : int   = 8
    length_penalty   : float = 1.3
    early_stopping   : bool  = True
    repetition_penalty:float = 1.2

    # multi-temperature sampling (key LB35.9 feature)
    use_sampling         : bool        = True
    sample_temperatures  : List[float] = field(
        default_factory=lambda: [0.60, 0.80, 1.05])
    num_sample_per_temp  : int         = 2
    mbr_top_p            : float       = 0.92

    @property
    def num_sample_cands(self) -> int:
        return len(self.sample_temperatures) * self.num_sample_per_temp

    # MBR weights (from LB35.9 — weighted combination)
    mbr_pool_cap  : int   = 32
    mbr_w_chrf    : float = 0.55
    mbr_w_bleu    : float = 0.25
    mbr_w_jaccard : float = 0.20
    mbr_w_length  : float = 0.10

    use_mixed_precision      : bool = True
    use_better_transformer   : bool = True
    use_bucket_batching      : bool = True
    use_adaptive_beams       : bool = True
    aggressive_postprocessing: bool = True
    checkpoint_freq          : int  = 200

    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.output_dir).mkdir(exist_ok=True, parents=True)
        if not torch.cuda.is_available():
            self.use_mixed_precision    = False
            self.use_better_transformer = False
        self.use_bf16_amp = bool(
            self.use_mixed_precision
            and self.device.type == "cuda"
            and _cuda_bf16_supported()
        )
        assert self.num_beams >= self.num_beam_cands


def setup_logging(output_dir):
    Path(output_dir).mkdir(exist_ok=True, parents=True)
    for h in logging.root.handlers[:]: logging.root.removeHandler(h)
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(Path(output_dir)/"ensemble.log"),
        ],
    )
    return logging.getLogger("ensemble")


def find_test_csv():
    candidates = [
        "/kaggle/input/deep-past-initiative-machine-translation/test.csv",
        "/kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv",
    ]
    for root, dirs, files in os.walk("/kaggle/input"):
        if "test.csv" in files:
            candidates.append(os.path.join(root, "test.csv"))
    for c in candidates:
        if os.path.exists(c): return c
    raise FileNotFoundError("test.csv not found")


def find_available_models(cfg):
    all_paths = [cfg.model_a_path, cfg.model_b_path, cfg.model_c_path]
    found = [p for p in all_paths if p and Path(p).exists()]
    if not found:
        print("Searching for T5 models...")
        for root, dirs, files in os.walk("/kaggle/input"):
            if "config.json" in files and "tokenizer_config.json" in files:
                try:
                    with open(os.path.join(root,"config.json")) as f:
                        cj = json.load(f)
                    arch  = str(cj.get("architectures",[""])[0]).lower()
                    mtype = str(cj.get("model_type","")).lower()
                    if "t5" in arch or "t5" in mtype:
                        found.append(root)
                        print(f"  Found: {root}")
                except: pass
    if not found:
        raise RuntimeError("No ByT5 models found! Attach model datasets.")
    return found


cfg    = Config()
logger = setup_logging(cfg.output_dir)

available_models = find_available_models(cfg)
print(f"\nDevice   : {cfg.device}")
print(f"BF16 AMP : {cfg.use_bf16_amp}")
print(f"Models   : {len(available_models)}")
for m in available_models: print(f"  {m}")
print(f"\nCandidates/model: beam×{cfg.num_beam_cands} + sample={cfg.num_sample_cands} ({len(cfg.sample_temperatures)} temps×{cfg.num_sample_per_temp}) = {cfg.num_beam_cands+cfg.num_sample_cands} total")
print(f"Total pool (~{len(available_models)} models): ~{len(available_models)*(cfg.num_beam_cands+cfg.num_sample_cands)} before dedup")



Device   : cuda
BF16 AMP : True
Models   : 2
  /kaggle/input/datasets/assiaben/final-byt5/byt5-akkadian-optimized-34x
  /kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1

Candidates/model: beam×4 + sample=6 (3 temps×2) = 10 total
Total pool (~2 models): ~20 before dedup


In [4]:
# ===========================================================================
# CELL 3 — Preprocessing (exactly from LB35.9 with KÙ.BABBAR fix)
# ===========================================================================

_V2    = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3    = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_GRAVE = str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})

def _ascii_to_diacritics(s: str) -> str:
    s = s.replace("sz","š").replace("SZ","Š")
    s = s.replace("s,","ṣ").replace("S,","Ṣ")
    s = s.replace("t,","ṭ").replace("T,","Ṭ")
    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)
    return s

_ALLOWED_FRACS = [
    (1/6,"0.16666"),(1/4,"0.25"),(1/3,"0.33333"),(1/2,"0.5"),
    (2/3,"0.66666"),(3/4,"0.75"),(5/6,"0.83333"),
]
_FRAC_TOL = 2e-3
_FLOAT_RE = re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")
_WS_RE    = re.compile(r"\s+")

def _canon_decimal(x: float) -> str:
    ip = int(math.floor(x+1e-12)); frac = x-ip
    best = min(_ALLOWED_FRACS, key=lambda t: abs(frac-t[0]))
    if abs(frac-best[0]) <= _FRAC_TOL:
        dec = best[1]
        if ip==0: return dec
        return f"{ip}{dec[1:]}" if dec.startswith("0.") else f"{ip}+{dec}"
    return f"{x:.5f}".rstrip("0").rstrip(".")

_GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I
)

def _normalize_gaps_vec(ser): return ser.str.replace(_GAP_UNIFIED_RE, "<gap>", regex=True)

_CHAR_TRANS = str.maketrans({
    "ḫ":"h","Ḫ":"H","ʾ":"",
    "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4",
    "₅":"5","₆":"6","₇":"7","₈":"8","₉":"9",
    "—":"-","–":"-",
})
_SUB_X = "ₓ"

_UNICODE_UPPER = r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_UNICODE_LOWER = r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"
_DET_UPPER_RE = re.compile(r"\(([" + _UNICODE_UPPER + r"0-9]{1,6})\)")
_DET_LOWER_RE = re.compile(r"\(([" + _UNICODE_LOWER + r"]{1,4})\)")

_KUBABBAR_RE   = re.compile(r"KÙ\.B\.")
_EXACT_FRAC_RE = re.compile(r"0\.8333|0\.6666|0\.3333|0\.1666|0\.625|0\.75|0\.25|0\.5")
_EXACT_FRAC_MAP = {
    "0.8333":"⅚","0.6666":"⅔","0.3333":"⅓","0.1666":"⅙",
    "0.625":"⅝","0.75":"¾","0.25":"¼","0.5":"½",
}
def _frac_repl(m): return _EXACT_FRAC_MAP[m.group(0)]


class OptimizedPreprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        ser = pd.Series(texts).fillna("").astype(str)
        ser = ser.apply(_ascii_to_diacritics)
        ser = ser.str.replace(_DET_UPPER_RE, r"\1",   regex=True)
        ser = ser.str.replace(_DET_LOWER_RE, r"{\1}", regex=True)
        ser = _normalize_gaps_vec(ser)
        ser = ser.str.translate(_CHAR_TRANS)
        ser = ser.str.replace(_SUB_X, "", regex=False)
        ser = ser.str.replace(_KUBABBAR_RE, "KÙ.BABBAR", regex=True)
        ser = ser.str.replace(_EXACT_FRAC_RE, _frac_repl, regex=True)
        ser = ser.str.replace(_FLOAT_RE, lambda m: _canon_decimal(float(m.group(1))), regex=True)
        ser = ser.str.replace(_WS_RE, " ", regex=True).str.strip()
        return ser.tolist()


print("Preprocessor ready (Unicode diacritics + KÙ.BABBAR fix)")


Preprocessor ready (Unicode diacritics + KÙ.BABBAR fix)


In [5]:
# ===========================================================================
# CELL 4 — Postprocessing (7 fixes from LB35.9)
# ===========================================================================

_SOFT_GRAM_RE = re.compile(
    r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)"
    r"(?:\.\s*(?:plur|plural|sing|singular))?"
    r"\.?\s*[^)]*\)", re.I
)
_BARE_GRAM_RE  = re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*", re.I)
_UNCERTAIN_RE  = re.compile(r"\(\?\)")
# FIX #4: curly quotes → straight (not deleted)
_CURLY_DQ_RE   = re.compile("[\u201c\u201d]")
_CURLY_SQ_RE   = re.compile("[\u2018\u2019]")
_MONTH_RE      = re.compile(r"\bMonth\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b", re.I)
_ROMAN2INT     = {"I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,"VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12}
_REPEAT_WORD   = re.compile(r"\b(\w+)(?:\s+\1\b)+")
_REPEAT_PUNCT  = re.compile(r"([.,])\1+")
_PUNCT_SPACE   = re.compile(r"\s+([.,:])")
_PN_RE         = re.compile(r"\bPN\b")

# FIX #3: parentheses PRESERVED — removed from forbidden
_FORBIDDEN_TRANS = str.maketrans("", "", '——<>⌈⌋⌊[]+ʾ;')

# FIX #2: commodity regex — requires space before hyphen
_COMMODITY_RE   = re.compile(r'(?<=\s)-(gold|tax|textiles)\b')
_COMMODITY_REPL = {"gold":"pašallum gold","tax":"šadduātum tax","textiles":"kutānum textiles"}
def _commodity_repl(m): return _COMMODITY_REPL[m.group(1)]

# FIX #1: shekel fraction corrections (host confirmed)
_SHEKEL_REPLS = [
    (re.compile(r'5\s+11\s*/\s*12\s+shekels?', re.I), '6 shekels less 15 grains'),
    (re.compile(r'5\s*/\s*12\s+shekels?',      re.I), '⅓ shekel 15 grains'),
    (re.compile(r'7\s*/\s*12\s+shekels?',      re.I), '½ shekel 15 grains'),
    (re.compile(r'1\s*/\s*12\s*(?:\(shekel\)|\bshekel)?', re.I), '15 grains'),
]

# FIX #6: safer slash-alternative regex
_SLASH_ALT_RE  = re.compile(r'(?<![0-9/])\s+/\s+(?![0-9])\S+')
_STRAY_MARKS   = re.compile(r'<<[^>]*>>|<(?!gap\b)[^>]*>')
_MULTI_GAP     = re.compile(r'(?:<gap>\s*){2,}')
# FIX #7: extra stray marks
_EXTRA_STRAY   = re.compile(r'(?<!\w)(?:\.\.|xx+)(?!\w)')
# FIX #5: ḫ/Ḫ in translations
_HACEK_TRANS   = str.maketrans({"ḫ":"h","Ḫ":"H"})

def _month_repl(m): return f"Month {_ROMAN2INT.get(m.group(1).upper(), m.group(1))}"


class VectorizedPostprocessor:
    def postprocess_batch(self, translations: List[str]) -> List[str]:
        s = pd.Series(translations).fillna("").astype(str)
        s = _normalize_gaps_vec(s)
        s = s.str.replace(_PN_RE,         "<gap>",          regex=True)
        s = s.str.replace(_COMMODITY_RE,  _commodity_repl,  regex=True)
        for pat, repl in _SHEKEL_REPLS:
            s = s.str.replace(pat, repl, regex=True)
        s = s.str.replace(_EXACT_FRAC_RE, _frac_repl,       regex=True)
        s = s.str.replace(_FLOAT_RE, lambda m: _canon_decimal(float(m.group(1))), regex=True)
        s = s.str.replace(_SOFT_GRAM_RE,  " ",               regex=True)
        s = s.str.replace(_BARE_GRAM_RE,  " ",               regex=True)
        s = s.str.replace(_UNCERTAIN_RE,  "",                regex=True)
        s = s.str.replace(_STRAY_MARKS,   "",                regex=True)
        s = s.str.replace(_EXTRA_STRAY,   "",                regex=True)
        s = s.str.replace(_SLASH_ALT_RE,  "",                regex=True)
        s = s.str.replace(_CURLY_DQ_RE,   '"',               regex=True)
        s = s.str.replace(_CURLY_SQ_RE,   "'",               regex=True)
        s = s.str.replace(_MONTH_RE,      _month_repl,       regex=True)
        s = s.str.replace(_MULTI_GAP,     "<gap>",           regex=True)
        # protect <gap> before stripping
        s = s.str.replace("<gap>", "\x00GAP\x00", regex=False)
        s = s.str.translate(_FORBIDDEN_TRANS)
        s = s.str.replace("\x00GAP\x00", " <gap> ", regex=False)
        s = s.str.translate(_HACEK_TRANS)
        s = s.str.replace(_REPEAT_WORD,  r"\1", regex=True)
        for n in range(4,1,-1):
            pat = r"\b((?:\w+\s+){"+str(n-1)+r"}\w+)(?:\s+\1\b)+"
            s = s.str.replace(pat, r"\1", regex=True)
        s = s.str.replace(_PUNCT_SPACE,  r"\1", regex=True)
        s = s.str.replace(_REPEAT_PUNCT, r"\1", regex=True)
        s = s.str.replace(_WS_RE, " ", regex=True).str.strip()
        return s.tolist()


preprocessor  = OptimizedPreprocessor()
postprocessor = VectorizedPostprocessor()
print("Postprocessor ready (7 fixes: parentheses preserved, shekel fractions, curly quotes, etc.)")



Postprocessor ready (7 fixes: parentheses preserved, shekel fractions, curly quotes, etc.)


In [6]:
# ===========================================================================
# CELL 5 — Dataset and bucketed sampler
# ===========================================================================

class AkkadianDataset(Dataset):
    def __init__(self, df, preprocessor, logger):
        self.sample_ids  = df["id"].tolist()
        proc             = preprocessor.preprocess_batch(df["transliteration"].tolist())
        self.input_texts = ["translate Akkadian to English: " + t for t in proc]
        logger.info(f"Dataset: {len(self.sample_ids)} samples")
        logger.info(f"  Sample: {self.input_texts[0][:100]}")

    def __len__(self):          return len(self.sample_ids)
    def __getitem__(self, idx): return self.sample_ids[idx], self.input_texts[idx]


class BucketBatchSampler(Sampler):
    def __init__(self, dataset, batch_size, num_buckets, logger, shuffle=False):
        self.batch_size = batch_size; self.shuffle = shuffle
        lengths    = [len(t.split()) for _, t in dataset]
        sorted_idx = sorted(range(len(lengths)), key=lambda i: lengths[i])
        bsz        = max(1, len(sorted_idx)//max(1,num_buckets))
        self.buckets = [
            sorted_idx[i*bsz: None if i==num_buckets-1 else (i+1)*bsz]
            for i in range(num_buckets)
        ]
        for i, b in enumerate(self.buckets):
            if b:
                bl = [lengths[x] for x in b]
                logger.info(f"  Bucket {i}: {len(b)} samples, len [{min(bl)},{max(bl)}]")

    def __iter__(self):
        for b in self.buckets:
            b = list(b)
            if self.shuffle: random.shuffle(b)
            for i in range(0,len(b),self.batch_size): yield b[i:i+self.batch_size]

    def __len__(self):
        return sum(math.ceil(len(b)/self.batch_size) for b in self.buckets)


print("Dataset & sampler ready.")


Dataset & sampler ready.


In [7]:
# ===========================================================================
# CELL 6 — Weighted MBR selector (from LB35.9)
# Key upgrade: chrF++×0.55 + BLEU×0.25 + Jaccard×0.20 + length_bonus×0.10
# ===========================================================================

class MBRSelector:
    def __init__(self, pool_cap=32, w_chrf=0.55, w_bleu=0.25,
                 w_jaccard=0.20, w_length=0.10):
        self._chrf_metric = sacrebleu.metrics.CHRF(word_order=2)
        self._bleu_metric = sacrebleu.metrics.BLEU(effective_order=True)
        self.pool_cap     = pool_cap
        self.w_chrf       = w_chrf
        self.w_bleu       = w_bleu
        self.w_jaccard    = w_jaccard
        self.w_length     = w_length
        self._pw_total    = max(w_chrf + w_bleu + w_jaccard, 1e-9)

    def _chrfpp(self, a, b):
        if not a or not b: return 0.0
        return float(self._chrf_metric.sentence_score(a, [b]).score)

    def _bleu(self, a, b):
        if not a or not b: return 0.0
        try:    return float(self._bleu_metric.sentence_score(a, [b]).score)
        except: return 0.0

    @staticmethod
    def _jaccard(a, b):
        ta, tb = set(a.lower().split()), set(b.lower().split())
        if not ta and not tb: return 100.0
        if not ta or not tb:  return 0.0
        return 100.0 * len(ta & tb) / len(ta | tb)

    def _pairwise(self, a, b):
        s = (self.w_chrf    * self._chrfpp(a, b)
           + self.w_bleu    * self._bleu(a, b)
           + self.w_jaccard * self._jaccard(a, b))
        return s / self._pw_total

    @staticmethod
    def _length_bonus(lengths, idx):
        if not lengths: return 100.0
        median = float(np.median(lengths))
        sigma  = max(median * 0.4, 5.0)
        z      = (lengths[idx] - median) / sigma
        return 100.0 * math.exp(-0.5 * z * z)

    @staticmethod
    def _dedup(xs):
        seen, out = set(), []
        for x in xs:
            x = str(x).strip()
            if x and x not in seen: out.append(x); seen.add(x)
        return out

    def pick(self, candidates: List[str]) -> str:
        cands = self._dedup(candidates)
        if self.pool_cap: cands = cands[:self.pool_cap]
        n = len(cands)
        if n == 0: return ""
        if n == 1: return cands[0]

        lengths = [len(c.split()) for c in cands]
        scores  = []
        for i in range(n):
            pw = sum(self._pairwise(cands[i], cands[j])
                     for j in range(n) if j != i) / max(1, n-1)
            lb = self._length_bonus(lengths, i)
            scores.append(pw + self.w_length * lb)
        return cands[int(np.argmax(scores))]


mbr = MBRSelector(
    pool_cap  = cfg.mbr_pool_cap,
    w_chrf    = cfg.mbr_w_chrf,
    w_bleu    = cfg.mbr_w_bleu,
    w_jaccard = cfg.mbr_w_jaccard,
    w_length  = cfg.mbr_w_length,
)
print(f"MBR: chrF++×{cfg.mbr_w_chrf} + BLEU×{cfg.mbr_w_bleu} + Jaccard×{cfg.mbr_w_jaccard} + length×{cfg.mbr_w_length}")


MBR: chrF++×0.55 + BLEU×0.25 + Jaccard×0.2 + length×0.1


In [8]:
# ===========================================================================
# CELL 7 — Model wrapper with multi-temperature sampling
# ===========================================================================

class ModelWrapper:
    def __init__(self, model_path, cfg: Config, logger, label):
        self.cfg = cfg; self.logger = logger; self.label = label
        logger.info(f"[{label}] Loading from {model_path}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
        self.model     = AutoModelForSeq2SeqLM.from_pretrained(
            model_path, local_files_only=True).to(cfg.device).eval()

        if cfg.device.type == "cuda":
            try: torch.set_float32_matmul_precision("high")
            except: pass

        n = sum(p.numel() for p in self.model.parameters())
        logger.info(f"[{label}] {n:,} parameters")

        if cfg.use_better_transformer and cfg.device.type == "cuda":
            try:
                from optimum.bettertransformer import BetterTransformer
                self.model = BetterTransformer.transform(self.model)
                logger.info(f"[{label}] BetterTransformer ON")
            except Exception as e:
                logger.warning(f"[{label}] BetterTransformer skipped: {e}")

    def collate(self, batch):
        ids = [s[0] for s in batch]
        enc = self.tokenizer(
            [s[1] for s in batch],
            max_length=self.cfg.max_input_length,
            padding=True, truncation=True, return_tensors="pt",
        )
        return ids, enc

    def generate_candidates(self, input_ids, attn, beam_size) -> List[List[str]]:
        cfg = self.cfg; B = input_ids.shape[0]
        Rb = cfg.num_beam_cands; Rs = cfg.num_sample_per_temp

        with _bf16_ctx(cfg.device, cfg.use_bf16_amp):
            # standard beam
            beam_out   = self.model.generate(
                input_ids=input_ids, attention_mask=attn,
                do_sample=False, num_beams=max(beam_size, Rb),
                num_return_sequences=Rb,
                max_new_tokens=cfg.max_new_tokens,
                length_penalty=cfg.length_penalty,
                early_stopping=cfg.early_stopping,
                repetition_penalty=cfg.repetition_penalty,
                use_cache=True,
            )
            beam_texts = self.tokenizer.batch_decode(beam_out, skip_special_tokens=True)

            # multi-temperature sampling
            all_samp_texts = []
            num_temps = 0
            if cfg.use_sampling and cfg.sample_temperatures:
                num_temps = len(cfg.sample_temperatures)
                for temp in cfg.sample_temperatures:
                    try:
                        samp_out = self.model.generate(
                            input_ids=input_ids, attention_mask=attn,
                            do_sample=True, num_beams=1,
                            top_p=cfg.mbr_top_p, temperature=temp,
                            num_return_sequences=Rs,
                            max_new_tokens=cfg.max_new_tokens,
                            repetition_penalty=cfg.repetition_penalty,
                            use_cache=True,
                        )
                        all_samp_texts.extend(
                            self.tokenizer.batch_decode(samp_out, skip_special_tokens=True))
                    except Exception as e:
                        self.logger.warning(f"[{self.label}] temp={temp} failed: {e}")
                        all_samp_texts.extend([""] * (B * Rs))

        pools = []
        for i in range(B):
            pool = list(beam_texts[i*Rb:(i+1)*Rb])
            if all_samp_texts and num_temps > 0:
                for t_idx in range(num_temps):
                    seg = t_idx * B * Rs + i * Rs
                    pool.extend(all_samp_texts[seg:seg+Rs])
            pools.append(pool)

        self.logger.info(
            f"[{self.label}] Pool: beam={Rb} + sample={num_temps}×{Rs}={num_temps*Rs} = {len(pools[0])} total")
        return pools

    def adaptive_beams(self, attn):
        if not self.cfg.use_adaptive_beams: return self.cfg.num_beams
        med   = float(attn.sum(dim=1).float().median().item())
        short = max(self.cfg.num_beam_cands, self.cfg.num_beams//2)
        return short if med < 100 else self.cfg.num_beams

    def unload(self):
        try:
            from optimum.bettertransformer import BetterTransformer
            self.model = BetterTransformer.reverse(self.model)
        except: pass
        del self.model, self.tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache(); torch.cuda.synchronize()
            free = (torch.cuda.get_device_properties(0).total_memory
                    - torch.cuda.memory_allocated()) / 1e9
            self.logger.info(f"[{self.label}] Unloaded. GPU free: {free:.2f} GB")


print("Model wrapper ready (multi-temperature sampling).")



Model wrapper ready (multi-temperature sampling).


In [9]:
# ===========================================================================
# CELL 8 — Run inference
# ===========================================================================

test_path = find_test_csv()
logger.info(f"Test data: {test_path}")
test_df   = pd.read_csv(test_path, encoding="utf-8")
logger.info(f"Test rows: {len(test_df)}")
print(f"Test rows : {len(test_df)}")

dataset    = AkkadianDataset(test_df, preprocessor, logger)
all_pools  = {}

logger.info("="*60)
logger.info(f"Ensemble × MBR  |  Cross-model candidate pooling")
for i, m in enumerate(available_models, 1):
    logger.info(f"  Model {i}: {m}")
logger.info(f"  Beam cands    : {cfg.num_beam_cands} (num_beams={cfg.num_beams})")
logger.info(f"  Sample cands  : {cfg.num_sample_cands} ({len(cfg.sample_temperatures)} temps×{cfg.num_sample_per_temp})")
logger.info(f"  MBR pool cap  : {cfg.mbr_pool_cap}")
logger.info(f"  BF16 AMP      : {cfg.use_bf16_amp}")
logger.info("="*60)

for i, model_path in enumerate(available_models, 1):
    label   = f"Model-{i}"
    wrapper = ModelWrapper(model_path, cfg, logger, label)

    if cfg.use_bucket_batching:
        sampler = BucketBatchSampler(dataset, cfg.batch_size, cfg.num_buckets, logger)
        dl = DataLoader(dataset, batch_sampler=sampler,
                        num_workers=cfg.num_workers,
                        pin_memory=(cfg.device.type=="cuda"),
                        collate_fn=wrapper.collate)
    else:
        dl = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=False,
                        num_workers=cfg.num_workers,
                        pin_memory=(cfg.device.type=="cuda"),
                        collate_fn=wrapper.collate)

    with torch.inference_mode():
        for batch_ids, enc in tqdm(dl, desc=f"  [{label}]"):
            ids_t = enc.input_ids.to(cfg.device, non_blocking=True)
            att_t = enc.attention_mask.to(cfg.device, non_blocking=True)
            beams = wrapper.adaptive_beams(att_t)
            try:
                pools = wrapper.generate_candidates(ids_t, att_t, beams)
                for sid, pool in zip(batch_ids, pools):
                    all_pools.setdefault(str(sid), []).extend(pool)
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    logger.error(f"OOM [{label}] — skipping batch")
                    torch.cuda.empty_cache()
                    for sid in batch_ids: all_pools.setdefault(str(sid), [])
                else: raise
            except Exception as e:
                logger.error(f"[{label}] batch error: {e}")
                for sid in batch_ids: all_pools.setdefault(str(sid), [])
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    wrapper.unload()

print(f"Inference done. Pools: {len(all_pools)}")


2026-03-18 19:08:57,733 | INFO | Test data: /kaggle/input/competitions/deep-past-initiative-machine-translation/test.csv
2026-03-18 19:08:57,750 | INFO | Test rows: 4
2026-03-18 19:08:57,762 | INFO | Dataset: 4 samples
2026-03-18 19:08:57,762 | INFO |   Sample: translate Akkadian to English: um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il<gap> da-tim aí-ip-ri-ni kà-a
2026-03-18 19:08:57,763 | INFO | ============================================================
2026-03-18 19:08:57,764 | INFO | Ensemble × MBR  |  Cross-model candidate pooling
2026-03-18 19:08:57,765 | INFO |   Model 1: /kaggle/input/datasets/assiaben/final-byt5/byt5-akkadian-optimized-34x
2026-03-18 19:08:57,767 | INFO |   Model 2: /kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1
2026-03-18 19:08:57,768 | INFO |   Beam cands    : 4 (num_beams=8)
2026-03-18 19:08:57,769 | INFO |   Sample cands  : 6 (3 temps×2)
2026-03-18 19:08:57,770 | INFO |   MBR pool cap  : 32
2026-03-18 19:08:57,770 | INFO |   BF16 AMP

Test rows : 4


Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
2026-03-18 19:09:21,720 | INFO | [Model-1] 581,653,248 parameters
2026-03-18 19:09:21,721 | WARNING | [Model-1] BetterTransformer skipped: No module named 'optimum'
2026-03-18 19:09:21,722 | INFO |   Bucket 0: 1 samples, len [20,20]
2026-03-18 19:09:21,723 | INFO |   Bucket 1: 1 samples, len [20,20]
2026-03-18 19:09:21,724 | INFO |   Bucket 2: 1 samples, len [23,23]
2026-03-18 19:09:21,724 | INFO |   Bucket 3: 1 samples, len [38,38]


  [Model-1]:   0%|          | 0/4 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/252 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
2026-03-18 19:10:55,432 | INFO | [Model-2] 581,653,248 parameters
2026-03-18 19:10:55,434 | WARNING | [Model-2] BetterTransformer skipped: No module named 'optimum'
2026-03-18 19:10:55,434 | INFO |   Bucket 0: 1 samples, len [20,20]
2026-03-18 19:10:55,435 | INFO |   Bucket 1: 1 samples, len [20,20]
2026-03-18 19:10:55,436 | INFO |   Bucket 2: 1 samples, len [23,23]
2026-03-18 19:10:55,436 | INFO |   Bucket 3: 1 samples, len [38,38]


  [Model-2]:   0%|          | 0/4 [00:00<?, ?it/s]

2026-03-18 19:11:05,131 | INFO | [Model-2] Pool: beam=4 + sample=3×2=6 = 10 total
2026-03-18 19:11:16,210 | INFO | [Model-2] Pool: beam=4 + sample=3×2=6 = 10 total
2026-03-18 19:11:28,630 | INFO | [Model-2] Pool: beam=4 + sample=3×2=6 = 10 total
2026-03-18 19:11:48,716 | INFO | [Model-2] Pool: beam=4 + sample=3×2=6 = 10 total
2026-03-18 19:11:59,265 | INFO | [Model-2] Unloaded. GPU free: 15.63 GB


Inference done. Pools: 4


In [10]:
# ===========================================================================
# CELL 9 — MBR reranking + save submission.csv
# ===========================================================================

sample_ids = [str(s) for s in dataset.sample_ids]
results    = []

for sid in tqdm(sample_ids, desc="  MBR"):
    pool   = all_pools.get(sid, [])
    post   = postprocessor.postprocess_batch(pool) if pool else []
    chosen = mbr.pick(post)
    if not chosen or not chosen.strip():
        chosen = "The tablet is too damaged to translate."
    results.append((sid, chosen))

    if cfg.checkpoint_freq and len(results) % cfg.checkpoint_freq == 0:
        ckpt = Path(cfg.output_dir)/f"ckpt_{len(results)}.csv"
        pd.DataFrame(results, columns=["id","translation"]).to_csv(ckpt, index=False)
        logger.info(f"Checkpoint: {len(results)} → {ckpt}")

result_df = pd.DataFrame(results, columns=["id","translation"])

# validate
empty = result_df["translation"].str.strip().eq("").sum()
lens  = result_df["translation"].str.len()
logger.info("="*60)
logger.info(f"Empty     : {empty} ({100*empty/max(1,len(result_df)):.2f}%)")
logger.info(f"Len mean  : {lens.mean():.1f}  median: {lens.median():.1f}  min: {lens.min()}  max: {lens.max()}")
for idx in [0, len(result_df)//4, len(result_df)//2, 3*len(result_df)//4, len(result_df)-1]:
    row = result_df.iloc[min(idx, len(result_df)-1)]
    logger.info(f"  ID {row['id']}: {str(row['translation'])[:80]}")
logger.info("="*60)

out_path = Path(cfg.output_dir)/"submission.csv"
result_df.to_csv(out_path, index=False)
logger.info(f"Saved → {out_path}  ({len(result_df)} rows)")

# save config
with open(Path(cfg.output_dir)/"config.json","w") as f:
    json.dump({
        "models"          : available_models,
        "num_beam_cands"  : cfg.num_beam_cands,
        "num_beams"       : cfg.num_beams,
        "sample_temps"    : cfg.sample_temperatures,
        "num_per_temp"    : cfg.num_sample_per_temp,
        "mbr_pool_cap"    : cfg.mbr_pool_cap,
        "mbr_w_chrf"      : cfg.mbr_w_chrf,
        "mbr_w_bleu"      : cfg.mbr_w_bleu,
        "mbr_w_jaccard"   : cfg.mbr_w_jaccard,
        "mbr_w_length"    : cfg.mbr_w_length,
        "bf16_amp"        : cfg.use_bf16_amp,
    }, f, indent=2)

print(f"\nSubmission file: {out_path}")
print(f"Config saved to: {Path(cfg.output_dir)/'config.json'}")


  MBR:   0%|          | 0/4 [00:00<?, ?it/s]

2026-03-18 19:12:00,502 | INFO | ============================================================
2026-03-18 19:12:00,504 | INFO | Empty     : 0 (0.00%)
2026-03-18 19:12:00,505 | INFO | Len mean  : 170.8  median: 150.0  min: 134  max: 249
2026-03-18 19:12:00,506 | INFO |   ID 0: Thus kārum Kanesh, say to the <gap> dātum, our messenger, every single colony an
2026-03-18 19:12:00,507 | INFO |   ID 1: As for the tablet of the City, whoever buys meteoric iron, it is not indebted to
2026-03-18 19:12:00,507 | INFO |   ID 2: As soon as you hear our letter, there he has given either for anything to the pa
2026-03-18 19:12:00,508 | INFO |   ID 3: I sent our tablet to every single colony and the trading stations. Whether he so
2026-03-18 19:12:00,509 | INFO |   ID 3: I sent our tablet to every single colony and the trading stations. Whether he so
2026-03-18 19:12:00,511 | INFO | ============================================================
2026-03-18 19:12:00,519 | INFO | Saved → /kaggle/working/subm


Submission file: /kaggle/working/submission.csv
Config saved to: /kaggle/working/config.json


In [11]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    if "config.json" in files and "tokenizer_config.json" in files:
        print(root)

/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr-v2/pytorch/default/1
/kaggle/input/datasets/assiaben/final-byt5/byt5-akkadian-optimized-34x
